Alec REYNEN - 26677221

# UTS 37009 Risk Management — Assessment Task 2 (AT2)
## Portfolio Risk Assessment, Stressed VaR/ES & Risk Minimization Strategy

**Author:** Alec REYNEN  
**Date:** Spring Session 2026  
**Valuation Date:** 14 August 2026  
**Dataset:** `AT2StockPriceData.csv` (Historical daily stock prices from 03 Jan 2000 to 14 Aug 2026)

---

### Executive Overview & Project Objectives

This notebook implements a quantitative market risk assessment framework for a non-linear equity-option portfolio containing positions in **Microsoft Corporation (MSFT)** and **Intel Corporation (INTC)**. 

The primary objective is to evaluate, compare, and stress-test the **10-day 99% Value-at-Risk (\\(\text{VaR}_{0.99}\\))** and **10-day 99% Expected Shortfall (\\(\text{ES}_{0.99}\\))** under various parametric, non-parametric, and semi-parametric risk modeling techniques. Additionally, we analyze hedging strategies to minimize portfolio risk exposure.

---

### 1. Portfolio Composition & Market Assumptions

At the valuation date (\\(t = 0\\), **14 August 2026**), the portfolio consists of four positions:
1. **Long** 1 At-The-Money (ATM) European Call option on MSFT stock maturing in 3 months (\\(T_1 = 3/12 = 0.25\\) year);
2. **Short** \\(\eta_1\\) shares of MSFT stock (\\(S_1\\));
3. **Long** 1 At-The-Money (ATM) European Call option on INTC stock maturing in 6 months (\\(T_2 = 6/12 = 0.50\\) year);
4. **Short** \\(\eta_2\\) shares of INTC stock (\\(S_2\\)).

#### Key Market Parameters:
* **Risk Management Horizon:** \\(\Delta t = 10/250 = 0.04\\) year (\\(10\\) trading days).
* **Risk-Free Rate:** \\(r = 3.5\%\\) p.a. continuously compounded for all tenors.
* **Dividend Yield:** \\(q = 0\%\\) (neither stock pays dividends).
* **Base Portfolio Weights:** \\(\eta_1 = \eta_2 = 1\\) (until Section 4.4 optimization).
* **Pricing Model:** Black-Scholes-Merton (BSM) framework.

---

### 2. Notebook Structure

The notebook is divided into five core sections following the assessment criteria:

* **Section 4.1 — Data Preparation & Risk Factor Mapping:** Construction of 10-day overlapping log-returns, estimation of EWMA conditional covariance matrix (\\(\theta = 0.04\\)), Black-Scholes pricing and Greeks (\\(\Delta, \Gamma, \Theta\\)), and algebraic derivation of the Delta-Gamma loss operator \\(L_{t+1}^{\Delta,\Gamma} = -c_t - b_t^\top X_{t+1} - \frac{1}{2}X_{t+1}^\top B_t X_{t+1}\\).
* **Section 4.2 — Market Risk Modelling (BAU):**
  * *4.2.1 Analytical Approach:* Variance-Covariance method under bivariate Gaussian risk factor changes.
  * *4.2.2 Historical Simulation:* Standard HS and Age-Weighted HS (\\(\lambda = 0.94\\)), comparing Delta-Gamma approximations vs. Full Valuation.
  * *4.2.3 Monte Carlo Simulation:* Maximum Likelihood Estimation (MLE) fitting of a Student's \\(t\\) copula, tail dependence analysis (`pycop`), and simulation under \\(t\\)-copula vs. bivariate Normal distributions.
* **Section 4.3 — Stressed VaR & Stressed ES Calculations:** Re-estimation of risk measures during two historical financial crises:
  * *Stress Period 1:* US Great Recession / Global Financial Crisis (03 Dec 2007 – 30 Jun 2009).
  * *Stress Period 2:* COVID-19 Market Crash (30 Jan 2020 – 01 Feb 2021).
* **Section 4.4 — Simple Risk Minimization Strategy:** Analytical derivation of Delta-neutral position sizes (\\(\eta_1^*, \eta_2^*\\)) and constrained numerical VaR optimization under Analytical, Historical, and Monte Carlo Full Valuation models.
* **Section 4.5 — Summary & Conclusions:** Synthesis of risk metrics, model comparison, and critical evaluation of stress testing and hedging limitations.

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt

# Inline plotting configuration
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Libraries successfully imported.")

Libraries successfully imported.


# Section 4.1 Data Preparation and Risk Factor Mapping

In [3]:
# Parameters
data_path = "data/AT2StockPriceData.csv"

# Load Stock Price Data and explicitly parse dates
df_prices = pd.read_csv(data_path)
stock_symbols = [col for col in df_prices.columns if col != "Date"]
# In this case, stock_symbols should be ['MSFT', 'INTC'] based on the provided data.

In [4]:
# 1. Log returns and Covariance Matrix

# Convert Date column to datetime explicitly (dayfirst=True for DD/MM/YYYY)
if "Date" in df_prices.columns:
    df_prices["Date"] = pd.to_datetime(df_prices["Date"], dayfirst=True)
    df_prices = df_prices.set_index("Date").sort_index()

# Compute 10-day Overlapping Log-Returns: X_{t+10} = ln(S_{t+10}) - ln(S_t)
horizon_days = 10
df_returns = np.log(df_prices / df_prices.shift(horizon_days)).dropna()

# Empirical Means of 10-day returns
mu_hat = df_returns.mean().values  # [mu1, mu2]

print("\n--- Empirical 10-day Return Means ---")
print(f"{stock_symbols[0]} mean (mu_hat_1): {mu_hat[0]:.6f}")
print(f"{stock_symbols[1]} mean (mu_hat_2): {mu_hat[1]:.6f}")



# EWMA Covariance Matrix Estimation with theta = 0.04
theta = 0.04
returns_mat = df_returns.values
T_obs = len(returns_mat)

# De-mean returns for EWMA calculation
demeaned_returns = returns_mat - mu_hat

# Initialize EWMA matrix with sample covariance matrix
Sigma_ewma = np.cov(returns_mat, rowvar=False)

# Recursive EWMA update: Sigma_t = theta * (x_t @ x_t.T) + (1 - theta) * Sigma_{t-1}
for t in range(T_obs):
    x_t = demeaned_returns[t, :].reshape(-1, 1)
    Sigma_ewma = theta * (x_t @ x_t.T) + (1 - theta) * Sigma_ewma

print("\n--- EWMA Covariance Matrix (10-day Horizon) ---")
df_ewma = pd.DataFrame(Sigma_ewma, index=["MSFT", "INTC"], columns=["MSFT", "INTC"])
print(df_ewma.to_string())


--- Empirical 10-day Return Means ---
MSFT mean (mu_hat_1): 0.004019
INTC mean (mu_hat_2): 0.002082

--- EWMA Covariance Matrix (10-day Horizon) ---
          MSFT      INTC
MSFT  0.018161  0.000933
INTC  0.000933  0.026644


In [5]:
# 2. Annualised Volatilities

# Compute annualised volatilities (dt = 10 / 250 = 0.04 year)
dt = 10.0 / 250.0
sigma1_annual = np.sqrt(Sigma_ewma[0, 0] / dt)
sigma2_annual = np.sqrt(Sigma_ewma[1, 1] / dt)

print("--- Annualised Volatilities ---")
print(f"{stock_symbols[0]} Annualised Volatility (sigma_1): {sigma1_annual * 100:.2f}% ({sigma1_annual:.6f})")
print(f"{stock_symbols[1]} Annualised Volatility (sigma_2): {sigma2_annual * 100:.2f}% ({sigma2_annual:.6f})")

--- Annualised Volatilities ---
MSFT Annualised Volatility (sigma_1): 67.38% (0.673811)
INTC Annualised Volatility (sigma_2): 81.61% (0.816150)


In [6]:
# 3. Black-Scholes Pricing and Option Greeks

# Current stock prices at valuation date (14 Aug 2026)
valuation_date = df_prices.index[-1]
S1 = df_prices.loc[valuation_date, "MSFT"]
S2 = df_prices.loc[valuation_date, "INTC"]

def bsm_call_greeks(S, K, T, r, sigma):
    """
    Computes BSM Call Price, Delta, Gamma, and Theta for a non-dividend paying stock.
    """
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    delta = norm.cdf(d1)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    theta = -(S * norm.pdf(d1) * sigma) / (2.0 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)

    return price, delta, gamma, theta

# Market and Option Parameters
r = 0.035        # Risk-free rate (3.5% p.a.)
T1 = 3.0 / 12.0  # 3 months for MSFT Call (0.25 year)
T2 = 6.0 / 12.0  # 6 months for INTC Call (0.50 year)
K1 = S1          # ATM Strike = Current Spot Price
K2 = S2          # ATM Strike = Current Spot Price

# Compute Prices &amp; Greeks
C1, Delta1, Gamma1, Theta1 = bsm_call_greeks(S1, K1, T1, r, sigma1_annual)
C2, Delta2, Gamma2, Theta2 = bsm_call_greeks(S2, K2, T2, r, sigma2_annual)

# Summary Table
greeks_summary = pd.DataFrame({
    'Metric': [
        'Spot Price (S)',
        'Strike (K)',
        'Maturity (T in yrs)',
        'Ann. Volatility (sigma)',
        'Option Price (C)',
        'Delta',
        'Gamma',
        'Theta (annual)'
    ],
    f'{stock_symbols[0]} Call (3M)': [S1, K1, T1, sigma1_annual, C1, Delta1, Gamma1, Theta1],
    f'{stock_symbols[1]} Call (6M)': [S2, K2, T2, sigma2_annual, C2, Delta2, Gamma2, Theta2]
})

print("\n--- Black-Scholes Prices &amp; Option Greeks ---")
print(greeks_summary.to_string(index=False))


--- Black-Scholes Prices &amp; Option Greeks ---
                 Metric  MSFT Call (3M)  INTC Call (6M)
         Spot Price (S)      494.469100      102.500000
             Strike (K)      494.469100      102.500000
    Maturity (T in yrs)        0.250000        0.500000
Ann. Volatility (sigma)        0.673811        0.816150
       Option Price (C)       68.034265       23.972813
                  Delta        0.577078        0.625090
                  Gamma        0.002350        0.006410
         Theta (annual)     -138.036338      -23.832409


## Task 4.1.4 — Analytical Derivation of the Delta-Gamma Loss Operator

### 1. Portfolio Value Function & Risk Factor Mapping

At time $t$, the portfolio value $V(t, z_1, z_2)$ is defined as a function of time $t$ and the log-price risk factors 
$z_1 = Z_{1,t} = \ln S_1(t)$ and $z_2 = Z_{2,t} = \ln S_2(t)$:

$$V(t, z_1, z_2) = C_1(t, e^{z_1}) - \eta_1 e^{z_1} + C_2(t, e^{z_2}) - \eta_2 e^{z_2}$$

where:
* $C_1(t, e^{z_1})$ is the Black-Scholes price of the 3-month ATM Call option on MSFT ($S_1 = e^{z_1}$)
* $C_2(t, e^{z_2})$ is the Black-Scholes price of the 6-month ATM Call option on INTC ($S_2 = e^{z_2}$)
* $\eta_1, \eta_2$ are the short position units in the underlying stocks MSFT and INTC, respectively, in our example

Over the risk management time horizon $[t, t+\Delta t]$, the random change in risk factors is given by the 10-day log-returns 
$X_{t+1} = (X_{1,t+1}, X_{2,t+1})^\top$, where $X_{i,t+1} = Z_{i,t+1} - Z_{i,t}$. 
The exact portfolio loss $L_{t+1}$ is defined as:

$$L_{t+1} := -(V_{t+1} - V_t) = -\left[ V(t+\Delta t, Z_{1,t} + X_{1,t+1}, Z_{2,t} + X_{2,t+1}) - V(t, Z_{1,t}, Z_{2,t}) \right]$$

---

### 2. Multivariate Taylor Series Expansion

To derive the Delta-Gamma loss approximation $L_{t+1}^{\Delta,\Gamma}$, we expand $V(t+\Delta t, z_1 + X_{1,t+1}, z_2 + X_{2,t+1})$ around $(t, z_1, z_2)$ using a second-order Taylor expansion with respect to the risk factors $z_1, z_2$ and a first-order expansion with respect to time $t$:

$$V_{t+1} - V_t \approx \frac{\partial V}{\partial t} \Delta t + \sum_{i=1}^2 \frac{\partial V}{\partial z_i} X_{i,t+1} + \frac{1}{2} \sum_{i=1}^2 \sum_{j=1}^2 \frac{\partial^2 V}{\partial z_i \partial z_j} X_{i,t+1} X_{j,t+1}$$

#### A. Sensitivity to Time (Theta):
Since the underlying stock positions $\eta_i e^{z_i}$ do not explicitly depend on time $t$:

$$\frac{\partial V}{\partial t} = \frac{\partial C_1}{\partial t} + \frac{\partial C_2}{\partial t} = \Theta_1 + \Theta_2$$

#### B. First-Order Sensitivities to Risk Factors (Delta):
Applying the chain rule with $\frac{d S_i}{d z_i} = \frac{d}{d z_i} (e^{z_i}) = e^{z_i} = S_i$:

$$\frac{\partial V}{\partial z_1} = \frac{\partial C_1}{\partial S_1} \frac{d S_1}{d z_1} - \eta_1 \frac{d S_1}{d z_1} = \Delta_1 S_1 - \eta_1 S_1 = (\Delta_1 - \eta_1) S_1$$

$$\frac{\partial V}{\partial z_2} = \frac{\partial C_2}{\partial S_2} \frac{d S_2}{d z_2} - \eta_2 \frac{d S_2}{d z_2} = \Delta_2 S_2 - \eta_2 S_2 = (\Delta_2 - \eta_2) S_2$$

#### C. Second-Order Sensitivities to Risk Factors (Gamma):
Differentiating $\frac{\partial V}{\partial z_i}$ with respect to $z_i$ using the product rule:

$$\frac{\partial^2 V}{\partial z_1^2} = \frac{\partial}{\partial z_1} \left[ (\Delta_1 - \eta_1) e^{z_1} \right] = \left( \frac{\partial \Delta_1}{\partial S_1} \frac{d S_1}{d z_1} \right) e^{z_1} + (\Delta_1 - \eta_1) e^{z_1} = \Gamma_1 S_1^2 + (\Delta_1 - \eta_1) S_1$$

$$\frac{\partial^2 V}{\partial z_2^2} = \frac{\partial}{\partial z_2} \left[ (\Delta_2 - \eta_2) e^{z_2} \right] = \left( \frac{\partial \Delta_2}{\partial S_2} \frac{d S_2}{d z_2} \right) e^{z_2} + (\Delta_2 - \eta_2) e^{z_2} = \Gamma_2 S_2^2 + (\Delta_2 - \eta_2) S_2$$

The cross-partial derivative is zero ($\frac{\partial^2 V}{\partial z_1 \partial z_2} = 0$) because each call option depends strictly on its own underlying stock price.

---

### 3. Canonical Representation of the Loss Operator

Substituting all partial derivatives back into the loss definition $L_{t+1}^{\Delta,\Gamma} = -(V_{t+1} - V_t)$, we obtain:

$$L_{t+1}^{\Delta,\Gamma} = -(\Theta_1 + \Theta_2)\Delta t - \sum_{i=1}^2 (\Delta_i - \eta_i) S_i X_{i,t+1} - \frac{1}{2} \sum_{i=1}^2 \left[ \Gamma_i S_i^2 + (\Delta_i - \eta_i) S_i \right] X_{i,t+1}^2$$

Expressing this relation in canonical matrix notation:

$$L_{t+1}^{\Delta,\Gamma} = -c_t - b_t^\top X_{t+1} - \frac{1}{2} X_{t+1}^\top B_t X_{t+1}$$

where:

1. **Scalar Constant ($c_t$):** Represents the deterministic theta time-decay over the 10-day horizon $\Delta t$:
   $$c_t = (\Theta_1 + \Theta_2) \Delta t$$

2. **Linear Sensitivity Vector ($b_t \in \mathbb{R}^2$):** Captures first-order delta exposures:
   $$b_t = \begin{bmatrix} (\Delta_1 - \eta_1) S_1 \\ (\Delta_2 - \eta_2) S_2 \end{bmatrix}$$

3. **Quadratic Sensitivity Matrix ($B_t \in \mathbb{R}^{2 \times 2}$):** Diagonal Hessian matrix combining second-order gamma convexity and log-normal return adjustments:
   $$B_t = \begin{bmatrix} \Gamma_1 S_1^2 + (\Delta_1 - \eta_1) S_1 & 0 \\ 0 & \Gamma_2 S_2^2 + (\Delta_2 - \eta_2) S_2 \end{bmatrix}$$

In [ ]:
# 5. Delta-Gamma Loss Operator Parameters

# Portfolio stock short positions
eta1 = 1.0
eta2 = 1.0

# Scalar constant term: c_t = (Theta1 + Theta2) * dt
c_t = (Theta1 + Theta2) * dt

# Linear vector: b_t = [(Delta1 - eta1)*S1, (Delta2 - eta2)*S2]^T
b_t = np.array([
    (Delta1 - eta1) * S1,
    (Delta2 - eta2) * S2
])

# Quadratic matrix: B_t (diagonal matrix)
B11 = Gamma1 * (S1**2) + (Delta1 - eta1) * S1
B22 = Gamma2 * (S2**2) + (Delta2 - eta2) * S2

B_t = np.array([
    [B11, 0.0],
    [0.0, B22]
])

print("\n--- Delta-Gamma Loss Operator Parameters ---")
print(f"\nConstant term (c_t): {c_t:.6f}")
print(f"\nLinear vector (b_t): [{b_t[0]:.6f}, {b_t[1]:.6f}]^T")
print("\nQuadratic matrix (B_t):")
df_Bt = pd.DataFrame(B_t, index=[stock_symbols[0], stock_symbols[1]], 
                     columns=[stock_symbols[0], stock_symbols[1]])
print(df_Bt.to_string())


--- Delta-Gamma Loss Operator Parameters ---
Constant term (c_t): -6.474750

Linear vector (b_t): [-209.121767, -38.428290]^T

Quadratic matrix (B_t):
            MSFT       INTC
MSFT  365.435057   0.000000
INTC    0.000000  28.915722


# Section 4.2 Market Risk Modelling

# Section 4.3 Stressed VaR and ES Calculations

# Section 4.4 A Simple Risk Minimizing Strategy